In [ ]:
nums = [1,2,4,6,8]
target = 5


for i in range(0,len(nums)):
    for j in range(0,len(nums)):
        if nums[i] + nums[j] == target:
        
            print(nums[i],nums[j])
        break


4 1


In [ ]:
nus = [1,3,5,2,7,8]
tar = 15

dic = {}

for i,j in enumerate (nus):
    min = tar - j
    if min in dic:
        print(dic[min],i)
    else:
        dic[j] = i

4 5


In [ ]:
def load_img():
    img = cv2.imread('/content/bricks.jpg')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

In [ ]:
# Complete Neural Network / Machine Learning Implementation from Scratch
import math
import random

# Seed for reproducibility
random.seed(42)

# --- 1. Matrix and Vector Math Utilities ---

class Matrix:
    def __init__(self, rows, cols, fill=0.0):
        self.rows = rows
        self.cols = cols
        self.data = [[fill for _ in range(cols)] for _ in range(rows)]

    @classmethod
    def random(cls, rows, cols):
        m = cls(rows, cols)
        for i in range(rows):
            for j in range(cols):
                m.data[i][j] = random.gauss(0.0, 1.0 * math.sqrt(2.0 / rows))
        return m

    def __matmul__(self, other):
        if self.cols != other.rows:
            raise ValueError(f"Incompatible dimensions for multiplication: {self.cols} vs {other.rows}")
        result = Matrix(self.rows, other.cols)
        for i in range(self.rows):
            for k in range(self.cols):
                vik = self.data[i][k]
                res_i = result.data[i]
                oth_k = other.data[k]
                for j in range(other.cols):
                    res_i[j] += vik * oth_k[j]
        return result

    def transpose(self):
        result = Matrix(self.cols, self.rows)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[j][i] = self.data[i][j]
        return result

    def apply(self, func):
        result = Matrix(self.rows, self.cols)
        for i in range(self.rows):
            for j in range(self.cols):
                result.data[i][j] = func(self.data[i][j])
        return result

# --- 2. Activation Functions ---

def relu(x):
    return max(0.0, x)

def relu_derivative(x):
    return 1.0 if x > 0.0 else 0.0

def sigmoid(x):
    if x < -709.0: return 0.0
    if x > 709.0: return 1.0
    return 1.0 / (1.0 + math.exp(-x))

# --- 3. Neural Network Layers ---

class DenseLayer:
    def __init__(self, input_size, output_size, activation="relu"):
        self.weights = Matrix.random(input_size, output_size)
        self.bias = Matrix(1, output_size, fill=0.0)
        self.activation = activation
        self.input = None
        self.output = None
        self.grad_weights = Matrix(input_size, output_size)
        self.grad_bias = Matrix(1, output_size)

    def forward(self, input_matrix):
        self.input = input_matrix
        # Z = Input @ Weights + Bias
        self.z = Matrix(input_matrix.rows, self.weights.cols)
        for i in range(input_matrix.rows):
            for j in range(self.weights.cols):
                val = self.bias.data[0][j]
                for k in range(input_matrix.cols):
                    val += input_matrix.data[i][k] * self.weights.data[k][j]
                self.z.data[i][j] = val
        
        # Apply activation
        self.output = Matrix(self.z.rows, self.z.cols)
        for i in range(self.z.rows):
            for j in range(self.z.cols):
                if self.activation == "relu":
                    self.output.data[i][j] = relu(self.z.data[i][j])
                elif self.activation == "sigmoid":
                    self.output.data[i][j] = sigmoid(self.z.data[i][j])
                else:
                    self.output.data[i][j] = self.z.data[i][j] # linear
        return self.output

    def backward(self, output_gradient, lr):
        # Compute derivative of activation
        dz = Matrix(self.z.rows, self.z.cols)
        for i in range(self.z.rows):
            for j in range(self.z.cols):
                if self.activation == "relu":
                    dz.data[i][j] = output_gradient.data[i][j] * relu_derivative(self.z.data[i][j])
                elif self.activation == "sigmoid":
                    sig = sigmoid(self.z.data[i][j])
                    dz.data[i][j] = output_gradient.data[i][j] * sig * (1.0 - sig)
                else:
                    dz.data[i][j] = output_gradient.data[i][j]

        # Gradients for weights and bias: dW = Input^T @ dz
        self.grad_weights = self.input.transpose() @ dz
        self.grad_bias = Matrix(1, dz.cols)
        for j in range(dz.cols):
            col_sum = sum(dz.data[i][j] for i in range(dz.rows))
            self.grad_bias.data[0][j] = col_sum

        # Gradient for previous layer: dx = dz @ Weights^T
        input_gradient = dz @ self.weights.transpose()

        # Update weights and bias (SGD step)
        for i in range(self.weights.rows):
            for j in range(self.weights.cols):
                self.weights.data[i][j] -= lr * self.grad_weights.data[i][j]
        for j in range(self.bias.cols):
            self.bias.data[0][j] -= lr * self.grad_bias.data[0][j]

        return input_gradient

# --- 4. Model and Training Loop ---

class SimpleMLP:
    def __init__(self, layer_sizes):
        self.layers = []
        for i in range(len(layer_sizes) - 1):
            act = "relu" if i < len(layer_sizes) - 2 else "sigmoid"
            self.layers.append(DenseLayer(layer_sizes[i], layer_sizes[i+1], activation=act))

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def train(self, X, y, epochs=100, lr=0.01):
        for epoch in range(epochs):
            total_loss = 0.0
            for r in range(X.rows):
                # Extract single row samples
                x_single = Matrix(1, X.cols)
                x_single.data[0] = list(X.data[r])
                y_single = Matrix(1, y.cols)
                y_single.data[0] = list(y.data[r])

                # Forward pass
                pred = self.forward(x_single)

                # Compute MSE Loss and gradient
                loss_grad = Matrix(1, pred.cols)
                for j in range(pred.cols):
                    diff = pred.data[0][j] - y_single.data[0][j]
                    total_loss += diff ** 2
                    loss_grad.data[0][j] = 2.0 * diff / pred.rows

                # Backward pass
                grad = loss_grad
                for layer in reversed(self.layers):
                    grad = layer.backward(grad, lr)

            if (epoch + 1) % 20 == 0 or epoch == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss / X.rows:.5f}")

# --- 5. Generate Synthetic Data and Run ---

if __name__ == "__main__":
    # Create XOR dataset
    X = Matrix(4, 2)
    X.data = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
    
    y = Matrix(4, 1)
    y.data = [[0.0], [1.0], [1.0], [0.0]]

    print("Initializing Neural Network (Architecture: 2 -> 4 -> 1)...")
    model = SimpleMLP([2, 4, 1])

    print("Starting training...")
    model.train(X, y, epochs=200, lr=0.1)

    print("\nTesting predictions after training:")
    for r in range(X.rows):
        x_single = Matrix(1, 2)
        x_single.data[0] = list(X.data[r])
        p = model.forward(x_single)
        print(f"Input: {X.data[r]} | Target: {y.data[r]} | Prediction: {p.data[0][j]:.4f}" if 'j' in locals() else f"Input: {X.data[r]} | Target: {y.data[r]} | Prediction: {p.data[0][0]:.4f}")
